In [128]:
import json
import glob
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import PatchTSTConfig, PatchTSTForPrediction


from sklearn.model_selection import train_test_split


# **DATA INGESTION**

In [129]:
match_files = glob.glob("/kaggle/input/fccricket/t20_master/*.json")

def load_match(path):
    with open(path) as f:
        return json.load(f)

matches = {}
for f in match_files:
    data = load_match(f)
    data["match_id"] = os.path.basename(f)
    matches[os.path.basename(f)] = data


In [130]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


# **BALL BY BALL**

In [131]:
def extract_balls(match_json):
    match_id = match_json["match_id"]
    meta = match_json["info"]

    rows = []
    for inning_i, inning in enumerate(match_json["innings"], start=1):
        batting_team = inning["team"]
        for over in inning["overs"]:
            over_number = over["over"]
            for ball in over["deliveries"]:
                row = {
                    "match_id": match_id,
                    "inning": inning_i,
                    "over": over_number,
                    "batter": ball.get("batter"),
                    "bowler": ball.get("bowler"),
                    "runs_batter": ball["runs"]["batter"],
                    "runs_extras": ball["runs"]["extras"],
                    "runs_total": ball["runs"]["total"],
                    "wicket_type": None,
                    "wicket_player": None,
                    "batting_team": batting_team,
                    "wicket_fielder": None,
                }

                if "wickets" in ball:
                    w = ball["wickets"][0]
                    row["wicket_type"] = w["kind"]
                    row["wicket_player"] = w["player_out"]
                    fielder_name = None

                    if "fielders" in w and len(w["fielders"]) > 0:
                        f = w["fielders"][0]

                        if isinstance(f, str):
                            fielder_name = f

                        elif isinstance(f, dict) and "name" in f:
                            fielder_name = f["name"]

                    if row["wicket_type"] == "caught and bowled":
                        if fielder_name is None:
                            fielder_name = row["bowler"]

                    row["wicket_fielder"] = fielder_name

                rows.append(row)
    return rows

In [132]:
all_rows = []
for name, match_json in matches.items():
    all_rows += extract_balls(match_json)

balls = pd.DataFrame(all_rows)

# **DATA HANDLING**

In [133]:
batting = balls.groupby(["match_id", "batter"]).agg(
    runs=("runs_batter", "sum"),
    balls_faced=("runs_batter", "count"),
    fours=("runs_batter", lambda x: (x == 4).sum()),
    sixes=("runs_batter", lambda x: (x == 6).sum()),
)


In [134]:
non_bowler_wickets = ['run out', 'retired hurt', 'retired out',  'obstructing the field']
bowling = balls.groupby(["match_id", "bowler"]).agg(
    runs_conceded=("runs_total", "sum"),
    balls_bowled=("runs_total", "count"),
    dot_balls = ("runs_total", lambda x : (x==0).sum()),
    wickets=("wicket_type", lambda x : (~x.isin(non_bowler_wickets) & x.notna()).sum()),
    lbw = ("wicket_type", lambda x : (x=='lbw').sum()),
    bowled = ("wicket_type", lambda x : (x=='bowled').sum()),
)


In [135]:
maidens = (
    balls.groupby(["match_id", "bowler", "inning", "over"])
         .runs_total.sum()
         .eq(0)
         .groupby(level=[0, 1])
         .sum()
)

bowling["maiden_overs"] = maidens.fillna(0).astype(int)

cols = list(bowling.columns)
cols.remove("maiden_overs")
insert_pos = cols.index("dot_balls") + 1
cols.insert(insert_pos, "maiden_overs")

bowling = bowling[cols]


In [136]:
fielding = balls.groupby(["match_id", "wicket_fielder"]).agg(
    f_catches=("wicket_type", lambda x : (x=='caught').sum()),
    f_caught_and_bowled=("wicket_type", lambda x : (x=='caught and bowled').sum()),
    f_stumpings=("wicket_type", lambda x : (x=='stumped').sum()),
    f_run_out = ("wicket_type", lambda x : (x=='run out').sum()),
)



In [137]:
battingn = batting.reset_index()
bowlingn = bowling.reset_index()
fieldingn = fielding.reset_index()

In [138]:
battingn = battingn.rename(columns={"batter": "player"})
bowlingn = bowlingn.rename(columns={"bowler": "player"})
fieldingn = fieldingn.rename(columns={"wicket_fielder": "player"})

player_match = pd.merge(battingn, bowlingn,
                        on=["match_id", "player"],
                        how="outer")

player_match = pd.merge(player_match, fieldingn,
                        on=["match_id", "player"],
                        how="outer")

player_match = player_match.fillna({
    "runs": 0,
    "balls_faced": 0,
    "fours": 0,
    "sixes": 0,
    "runs_conceded": 0,
    "balls_bowled": 0,
    "wickets": 0,
    "lbw":0,
    "bowled":0,
    "dot_balls":0,
    "maiden_overs":0,
    "f_catches": 0,
    "f_caught_and_bowled": 0,
    "f_stumpings": 0,
    "f_run_out": 0
})

In [139]:
player_match["match_id"] = (
    player_match["match_id"]
    .astype(str)
    .str.replace(".json", "", regex=False)
)

player_match["match_id"] = player_match["match_id"].astype(int)
player_match = player_match.sort_values("match_id", ascending=False).reset_index(drop=True)

In [140]:
player_match["strike_rate"] = (
    (player_match["runs"] * 100 / player_match["balls_faced"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)


player_match["overs_bowled"] = (player_match["balls_bowled"] / 6).astype(int)
player_match["economy"] = (
    (player_match["runs_conceded"] / player_match["overs_bowled"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

# **FANTASY POINTS**

In [141]:
def calculate_fantasy_points(df):

    df["batting_fp"] = 0.0
    df["bowing_fp"] = 0.0
    df["fielding_fp"] = 0.0
    df["fp"] = 0.0

    # Batting Points
    df["batting_fp"] += df["runs"] * 1
    df["batting_fp"] += df["fours"] * 4
    df["batting_fp"] += df["sixes"] * 6

    df["batting_fp"] += np.where(df["runs"] >= 100, 16,
                                np.where(df["runs"] >= 75, 12,
                                np.where(df["runs"] >= 50, 8,
                                np.where(df["runs"] >= 25, 4, 0))))

    df["batting_fp"] += np.where((df["runs"] == 0) & (df["balls_faced"] > 0), -2, 0)

    mask_sr = (df["balls_faced"] >= 10) & (df["balls_bowled"] == 0)

    sr_bonus_values = np.where(df["strike_rate"] > 170, 6,
                               np.where(df["strike_rate"] > 150, 4,
                               np.where(df["strike_rate"] >= 130, 2,
                               np.where(df["strike_rate"] >= 60, 0,
                               np.where(df["strike_rate"] >= 50, -2,
                               np.where(df["strike_rate"] >= 0, -4, 0))))))
    df.loc[mask_sr, "batting_fp"] += sr_bonus_values[mask_sr]


    # Bowling Points
    df["bowing_fp"] += df["wickets"] * 30

    df["bowing_fp"] += (df["lbw"] + df["bowled"]) * 8

    df["bowing_fp"] += np.where(df["wickets"] >= 5, 12,
                                np.where(df["wickets"] == 4, 8,
                                np.where(df["wickets"] == 3, 4, 0)))

    if "dot_balls" in df.columns:
        df["bowing_fp"] += df["dot_balls"] * 1

    if "maiden_overs" in df.columns:
        df["bowing_fp"] += df["maiden_overs"] * 12

    mask_economy = df["overs_bowled"] >= 2

    economy_bonus_values = np.where( df["economy"] < 5, 6,
                                           np.where(df["economy"] < 6, 4,
                                           np.where(df["economy"] < 7, 2,
                                           np.where(df["economy"] >= 12, -6,
                                           np.where(df["economy"] >= 11, -4,
                                           np.where(df["economy"] >= 10, -2, 0))))))
    df.loc[mask_economy, "bowing_fp"] += economy_bonus_values[mask_economy]


    #Fielding Points

    df["fielding_fp"] += df["f_catches"] * 8
    df["fielding_fp"] += df["f_caught_and_bowled"] * 8
    df["fielding_fp"] += df["f_stumpings"] * 12
    df["fielding_fp"] += df["f_run_out"] * 6

    df["fielding_fp"] += np.where(df["f_catches"] >= 3, 4, 0)

    df.drop(columns=["overs_bowled"], inplace=True, errors='ignore')
    df["fp"] = df["batting_fp"] + df["bowing_fp"] + df["fielding_fp"]

    return df

In [142]:
player_match = calculate_fantasy_points(player_match)

In [143]:
player_match

,match_id,player,runs,balls_faced,fours,sixes,runs_conceded,balls_bowled,dot_balls,maiden_overs,...,f_catches,f_caught_and_bowled,f_stumpings,f_run_out,strike_rate,economy,batting_fp,bowing_fp,fielding_fp,fp
0,1512000,Asif Javed,0.0,0.0,0.0,0.0,12.0,20.0,10.0,1.0,...,1.0,0.0,0.0,0.0,0.000000,4.000000,0.0,58.0,8.0,66.0
1,1512000,Y Saranonnakkun,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,-2.0,0.0,0.0,-2.0
2,1512000,N Nuntarach,2.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,25.000000,0.000000,2.0,0.0,0.0,2.0
3,1512000,Ahmer Bin Nisar,0.0,0.0,0.0,0.0,6.0,6.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,6.000000,0.0,31.0,0.0,31.0
4,1512000,Asif Shaikh,0.0,0.0,0.0,0.0,18.0,19.0,8.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,6.000000,0.0,40.0,0.0,40.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140241,211028,D Gough,0.0,0.0,0.0,0.0,17.0,19.0,12.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,5.666667,0.0,110.0,0.0,110.0
140242,211028,DR Martyn,4.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,100.000000,0.000000,8.0,0.0,0.0,8.0
140243,211028,GD McGrath,5.0,12.0,0.0,0.0,31.0,25.0,10.0,0.0,...,0.0,0.0,0.0,0.0,41.666667,7.750000,5.0,104.0,0.0,109.0
140244,211028,GO Jones,19.0,14.0,4.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,135.714286,0.000000,37.0,0.0,8.0,45.0


# SEQUENCE

In [144]:
for df in [battingn, bowlingn, fieldingn]:
    df['match_id'] = (
        df['match_id']
        .str.replace('.json', '', regex=False)
        .astype(int)
    )

In [145]:
for df in [battingn, bowlingn, fieldingn]:
    df.sort_values(by = 'match_id', inplace=True)

In [146]:
BAT_COLS   = ['runs', 'balls_faced', 'fours', 'sixes']          # (4)
BOWL_COLS  = ['runs_conceded','balls_bowled','dot_balls','wickets',
              'lbw','bowled','maiden_overs']                         #(7)
FIELD_COLS = ['f_catches','f_caught_and_bowled','f_stumpings','f_run_out']           # (4)

In [147]:
def time_split(df, ratio=0.8):
    match_ids = (
        df[['match_id']]
        .drop_duplicates()
        .sort_values('match_id')['match_id']
        .tolist()
    )
    idx = int(len(match_ids) * ratio)
    return set(match_ids[:idx]), set(match_ids[idx:])

train_matches, test_matches = time_split(battingn)


In [148]:
def extract_k_to_i(df, player, mid, cols, K, mode,
                   train_matches, test_matches):

    history = df[
        (df['player'] == player) &
        (df['match_id'] < mid)
    ].sort_values('match_id')

    if len(history) < K:
        return None

    hist_ids = history['match_id'].tail(K)

    if mode == 'train':
        if mid not in train_matches:
            return None
        if not hist_ids.isin(train_matches).all():
            return None

    if mode == 'test':
        if mid not in test_matches:
            return None

    X = history[cols].tail(K).to_numpy(np.float32)
    y = df.loc[
        (df['player'] == player) &
        (df['match_id'] == mid),
        cols
    ].iloc[0].to_numpy(np.float32)

    return X, y


In [149]:
def build_xy(df, cols, K, mode, train_matches, test_matches):
    
    X, y = [], []

    if mode == "train":
        valid_matches = set(train_matches)
    else:
        valid_matches = set(test_matches)

    for player, pdf in df.groupby("player"):
        pdf = pdf.sort_values("match_id")

        feats = pdf[cols].to_numpy(np.float32)
        mids  = pdf["match_id"].to_numpy()

        for i in range(K, len(pdf)):
            if mids[i] not in valid_matches:
                continue

            X.append(feats[i-K:i])
            y.append(feats[i])

    if len(X) == 0:
        return None, None

    return np.stack(X), np.stack(y)


In [150]:
K = 10

bat_X_tr, bat_y_tr = build_xy(player_match, BAT_COLS, K, 'train', train_matches, test_matches)
bat_X_te, bat_y_te = build_xy(player_match, BAT_COLS, K, 'test',  train_matches, test_matches)

bowl_X_tr, bowl_y_tr = build_xy(player_match, BOWL_COLS, K, 'train', train_matches, test_matches)
bowl_X_te, bowl_y_te = build_xy(player_match, BOWL_COLS, K, 'test',  train_matches, test_matches)

field_X_tr, field_y_tr = build_xy(player_match, FIELD_COLS, K, 'train', train_matches, test_matches)
field_X_te, field_y_te = build_xy(player_match, FIELD_COLS, K, 'test',  train_matches, test_matches)


In [151]:
def normalize_data(X, y, mean=None, std=None):
    if mean is None:
        mean = X.mean(axis=(0, 1), keepdims=True)
        std  = X.std(axis=(0, 1), keepdims=True) + 1e-6
    
    X_norm = (X - mean) / std
    
    # Normalize targets (y) to Z-scores
    mean_2d = mean.squeeze(1)
    std_2d = std.squeeze(1)
    y_norm = (y - mean_2d) / std_2d
    
    return X_norm, y_norm, mean, std

# Apply to Training Data
bat_X_tr, bat_y_tr, bat_mean, bat_std = normalize_data(bat_X_tr, bat_y_tr)
bowl_X_tr, bowl_y_tr, bowl_mean, bowl_std = normalize_data(bowl_X_tr, bowl_y_tr)
field_X_tr, field_y_tr, field_mean, field_std = normalize_data(field_X_tr, field_y_tr)

# Apply to Test Data
bat_X_te, bat_y_te, _, _ = normalize_data(bat_X_te, bat_y_te, bat_mean, bat_std)
bowl_X_te, bowl_y_te, _, _ = normalize_data(bowl_X_te, bowl_y_te, bowl_mean, bowl_std)
field_X_te, field_y_te, _, _ = normalize_data(field_X_te, field_y_te, field_mean, field_std)

In [152]:

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [153]:
BATCH_SIZE = 64

bat_train_loader = DataLoader(
    SeqDataset(bat_X_tr, bat_y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

bowl_train_loader = DataLoader(
    SeqDataset(bowl_X_tr, bowl_y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

field_train_loader = DataLoader(
    SeqDataset(field_X_tr, field_y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)


In [154]:
bat_config = PatchTSTConfig(
    context_length=K,
    prediction_length=1,
    num_input_channels=4
)

bat_model = PatchTSTForPrediction(bat_config)
bat_model = bat_model.to(device)


bowl_config = PatchTSTConfig(
    context_length=K,
    prediction_length=1,
    num_input_channels=7
)

bowl_model = PatchTSTForPrediction(bowl_config)
bowl_model = bowl_model.to(device)

field_config = PatchTSTConfig(
    context_length=K,
    prediction_length=1,
    num_input_channels=4
)

field_model = PatchTSTForPrediction(field_config)
field_model = field_model.to(device)


In [155]:
def train_patchtst_hf(model, loader, epochs, lr, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = torch.nn.SmoothL1Loss()

    for epoch in range(epochs):
        model.train()
        total = 0.0

        for X, y in loader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            outputs = model(past_values=X)
            preds = outputs.prediction_outputs.squeeze(1)

            loss = loss_fn(preds, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total/len(loader):.4f}")


In [156]:
train_patchtst_hf(bat_model, bat_train_loader, epochs=10, lr=1e-3, device=device)
train_patchtst_hf(bowl_model, bowl_train_loader, epochs=10, lr=1e-3, device=device)
train_patchtst_hf(field_model, field_train_loader, epochs=10, lr=1e-3, device=device)


Epoch 1/10 | Loss: 0.2943
Epoch 2/10 | Loss: 0.2939
Epoch 3/10 | Loss: 0.2940
Epoch 4/10 | Loss: 0.2937
Epoch 5/10 | Loss: 0.2935
Epoch 6/10 | Loss: 0.2935
Epoch 7/10 | Loss: 0.2932
Epoch 8/10 | Loss: 0.2932
Epoch 9/10 | Loss: 0.2930
Epoch 10/10 | Loss: 0.2929
Epoch 1/10 | Loss: 0.1882
Epoch 2/10 | Loss: 0.1874
Epoch 3/10 | Loss: 0.1871
Epoch 4/10 | Loss: 0.1869
Epoch 5/10 | Loss: 0.1869
Epoch 6/10 | Loss: 0.1868
Epoch 7/10 | Loss: 0.1867
Epoch 8/10 | Loss: 0.1868
Epoch 9/10 | Loss: 0.1868
Epoch 10/10 | Loss: 0.1866
Epoch 1/10 | Loss: 0.1962
Epoch 2/10 | Loss: 0.1958
Epoch 3/10 | Loss: 0.1957
Epoch 4/10 | Loss: 0.1954
Epoch 5/10 | Loss: 0.1956
Epoch 6/10 | Loss: 0.1952
Epoch 7/10 | Loss: 0.1952
Epoch 8/10 | Loss: 0.1954
Epoch 9/10 | Loss: 0.1954
Epoch 10/10 | Loss: 0.1952


In [157]:
SAVE_DIR = "/kaggle/working/models"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(bat_model.state_dict(),   f"{SAVE_DIR}/bat_model.pt")
torch.save(bowl_model.state_dict(),  f"{SAVE_DIR}/bowl_model.pt")
torch.save(field_model.state_dict(), f"{SAVE_DIR}/field_model.pt")


In [158]:
bat_model.to(device).eval()
bowl_model.to(device).eval()
field_model.to(device).eval()

PatchTSTForPrediction(
  (model): PatchTSTModel(
    (scaler): PatchTSTScaler(
      (scaler): PatchTSTStdScaler()
    )
    (patchifier): PatchTSTPatchify()
    (masking): Identity()
    (encoder): PatchTSTEncoder(
      (embedder): PatchTSTEmbedding(
        (input_embedding): Linear(in_features=1, out_features=128, bias=True)
      )
      (positional_encoder): PatchTSTPositionalEncoding(
        (positional_dropout): Identity()
      )
      (layers): ModuleList(
        (0-2): 3 x PatchTSTEncoderLayer(
          (self_attn): PatchTSTAttention(
            (k_proj): Linear(in_features=128, out_features=128, bias=True)
            (v_proj): Linear(in_features=128, out_features=128, bias=True)
            (q_proj): Linear(in_features=128, out_features=128, bias=True)
            (out_proj): Linear(in_features=128, out_features=128, bias=True)
          )
          (dropout_path1): Identity()
          (norm_sublayer1): PatchTSTBatchNorm(
            (batchnorm): BatchNorm1d(128, eps=

In [159]:
def get_player_history_padded(df, player, current_match_id, cols, mean, std):

    player_data = df.loc[[player]]
    history = player_data[player_data['match_id'] < current_match_id]
    
    data = history[cols].tail(K).to_numpy(np.float32)
    
    if len(data) < K:
        rows_needed = K - len(data)
        padding = np.zeros((rows_needed, len(cols)), dtype=np.float32)
        if len(data) > 0:
            data = np.vstack([padding, data])
        else:
            data = padding
            
    X_norm = (data - mean) / std
    
    return torch.tensor(X_norm, dtype=torch.float32).to(device)

In [160]:
player_match_sorted = player_match.sort_values(['player', 'match_id'])
pm_indexed = player_match_sorted.set_index('player')

In [161]:
pm_indexed

,match_id,runs,balls_faced,fours,sixes,runs_conceded,balls_bowled,dot_balls,maiden_overs,wickets,...,f_catches,f_caught_and_bowled,f_stumpings,f_run_out,strike_rate,economy,batting_fp,bowing_fp,fielding_fp,fp
player,,,,,,,,,,,,,,,,,,,,,
A Ahmadhel,1235832,2.0,8.0,0.0,0.0,6.0,5.0,3.0,0.0,1.0,...,0.0,0.0,0.0,0.0,25.000000,0.00,2.0,33.0,0.0,35.0
A Ahmadhel,1275269,4.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,66.666667,0.00,8.0,0.0,0.0,8.0
A Ahmadhel,1275271,0.0,7.0,0.0,0.0,15.0,7.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,15.00,-2.0,1.0,0.0,-1.0
A Ahmadhel,1443778,2.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,33.333333,0.00,2.0,0.0,8.0,10.0
A Alexander,1453919,0.0,0.0,0.0,0.0,19.0,24.0,12.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.000000,4.75,0.0,86.0,0.0,86.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zulqarnain Haider,1310162,0.0,0.0,0.0,0.0,19.0,24.0,12.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.000000,4.75,0.0,56.0,0.0,56.0
Zulqarnain Haider,1310164,0.0,0.0,0.0,0.0,4.0,12.0,9.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.000000,2.00,0.0,45.0,0.0,45.0
Zulqarnain Haider,1310165,3.0,3.0,0.0,0.0,8.0,6.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,100.000000,8.00,3.0,2.0,0.0,5.0


In [162]:
all_predictions = []

print("Starting Inference...")
with torch.no_grad():
    for match_id in list(test_matches):
        unique_players = set(player_match.loc[player_match['match_id'] == match_id, 'player'])
        
        for player in unique_players:
            row_pred = {'match_id': match_id, 'player': player}
            
            X_bat = get_player_history_padded(pm_indexed, player, match_id, BAT_COLS, bat_mean, bat_std)
            out = bat_model(past_values=X_bat).prediction_outputs 
            preds = (out.squeeze(1).cpu().numpy() * bat_std.squeeze()) + bat_mean.squeeze()
            preds = np.maximum(0, preds.flatten())
            for i, col in enumerate(BAT_COLS): row_pred[col] = preds[i]

            X_bowl = get_player_history_padded(pm_indexed, player, match_id, BOWL_COLS, bowl_mean, bowl_std)
            out = bowl_model(past_values=X_bowl).prediction_outputs
            preds = (out.squeeze(1).cpu().numpy() * bowl_std.squeeze()) + bowl_mean.squeeze()
            preds = np.maximum(0, preds.flatten())
            for i, col in enumerate(BOWL_COLS): row_pred[col] = preds[i]

            X_field = get_player_history_padded(pm_indexed, player, match_id, FIELD_COLS, field_mean, field_std)
            out = field_model(past_values=X_field).prediction_outputs
            preds = (out.squeeze(1).cpu().numpy() * field_std.squeeze()) + field_mean.squeeze()
            preds = np.maximum(0, preds.flatten())
            for i, col in enumerate(FIELD_COLS): row_pred[col] = preds[i]
            
            all_predictions.append(row_pred)

pred_df = pd.DataFrame(all_predictions)

Starting Inference...


In [163]:
pred_df

,match_id,player,runs,balls_faced,fours,sixes,runs_conceded,balls_bowled,dot_balls,wickets,lbw,bowled,maiden_overs,f_catches,f_caught_and_bowled,f_stumpings,f_run_out
0,1482819,A Kalasi,1.008823,1.063592,0.000000,0.089217,15.182002,19.375170,8.244390,0.651645,0.026412,0.036237,0.02117,0.110197,0.000000,0.0,0.00000
1,1482819,P Thongsa,4.151691,5.759720,0.313321,0.000000,3.173967,2.574808,0.751332,0.080735,0.000000,0.035748,0.00000,0.116715,0.000000,0.0,0.00000
2,1482819,A Parima,17.031738,14.075136,1.322268,0.694220,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.441255,0.000000,0.0,0.02526
3,1482819,N Salekar,9.198433,11.272949,0.432007,0.089217,23.837337,20.971642,9.024249,0.833699,0.000000,0.021169,0.00000,0.102538,0.060956,0.0,0.00000
4,1482819,T Parima,22.893372,16.034035,1.646987,1.264924,0.021228,0.045296,0.022714,0.000000,0.000000,0.000000,0.00000,0.368651,0.000000,0.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28055,1490890,Ali Nawaz,3.188090,2.588608,0.195666,0.191935,11.779654,7.715817,2.318317,0.498997,0.000000,0.071532,0.00000,0.000000,0.000000,0.0,0.00000
28056,1490890,Mustafa Omer,4.963987,5.999712,0.523463,0.084289,1.036673,0.679187,0.321726,0.000000,0.000000,0.000000,0.00000,0.030919,0.000000,0.0,0.00000
28057,1490890,C Roberts,16.507538,15.627436,1.751498,0.098421,0.500362,0.464667,0.285974,0.107246,0.035749,0.000000,0.00000,0.174668,0.000000,0.0,0.02526
28058,1490890,L Canessane,7.584845,7.879883,0.729159,0.095956,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.118266,0.000000,0.0,0.00000


In [164]:
pred_df = pd.DataFrame(all_predictions)

pred_df["strike_rate"] = (
    (pred_df["runs"] * 100 / pred_df["balls_faced"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

pred_df["overs_bowled"] = (pred_df["balls_bowled"] / 6).astype(int)
pred_df["economy"] = (
    (pred_df["runs_conceded"] / pred_df["overs_bowled"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

print("Derived metrics calculated.")

Derived metrics calculated.


In [165]:
final_pred_df = calculate_fantasy_points(pred_df)

print("Final Fantasy Points Calculated.")

Final Fantasy Points Calculated.


In [166]:
actual_df = player_match[player_match['match_id'].isin(test_matches)].copy()

actual_df = actual_df[['match_id', 'player'] + BAT_COLS + BOWL_COLS + FIELD_COLS + ['batting_fp', 'bowing_fp', 'fielding_fp', 'fp']]

comparison_df = pd.merge(
    final_pred_df, 
    actual_df, 
    on=['match_id', 'player'], 
    how='inner', 
    suffixes=('_pred', '_actual')
)

In [167]:
comparison_df.to_excel("fantasy_predictions.xlsx", index=False)

In [168]:
comparison_df.columns

Index(['match_id', 'player', 'runs_pred', 'balls_faced_pred', 'fours_pred',
       'sixes_pred', 'runs_conceded_pred', 'balls_bowled_pred',
       'dot_balls_pred', 'wickets_pred', 'lbw_pred', 'bowled_pred',
       'maiden_overs_pred', 'f_catches_pred', 'f_caught_and_bowled_pred',
       'f_stumpings_pred', 'f_run_out_pred', 'strike_rate', 'economy',
       'batting_fp_pred', 'bowing_fp_pred', 'fielding_fp_pred', 'fp_pred',
       'runs_actual', 'balls_faced_actual', 'fours_actual', 'sixes_actual',
       'runs_conceded_actual', 'balls_bowled_actual', 'dot_balls_actual',
       'wickets_actual', 'lbw_actual', 'bowled_actual', 'maiden_overs_actual',
       'f_catches_actual', 'f_caught_and_bowled_actual', 'f_stumpings_actual',
       'f_run_out_actual', 'batting_fp_actual', 'bowing_fp_actual',
       'fielding_fp_actual', 'fp_actual'],
      dtype='object')

In [172]:
check_df = comparison_df[['player', 'match_id', 'fp_pred', 'fp_actual']]

In [186]:
check_df

,player,match_id,fp_pred,fp_actual
0,A Kalasi,1482819,34.974684,35.0
1,P Thongsa,1482819,9.798058,8.0
2,A Parima,1482819,30.167732,93.0
3,N Salekar,1482819,46.974302,4.0
4,T Parima,1482819,40.042792,87.0
...,...,...,...,...
28055,Ali Nawaz,1490890,22.982858,56.0
28056,Mustafa Omer,1490890,8.132656,25.0
28057,C Roberts,1490890,29.442308,83.0
28058,L Canessane,1490890,12.023351,52.0


In [197]:
total=0
for matchid in check_df['match_id'] :

    set1 = set(check_df[check_df['match_id'] == matchid].sort_values(by='fp_pred', ascending=False).head(10)['player'])
    set2 = set(check_df[check_df['match_id'] == matchid].sort_values(by='fp_actual', ascending=False).head(10)['player'])
    total = total + len(set1.intersection(set2))

print(total/len(check_df['match_id']))

5.551888809693514
